![Original climate-globe illustration introducing Four windows on the tropical Pacific; illustrative, not a data map](assets/atlas-splash.png)

# 22 · Four windows on the tropical Pacific

**Guiding question:** Why does Niño 4 need two polygons on a longitude map?

**Data:** Real C3S/ECMWF ERA5 Climate Pulse CSV snapshots retrieved 2026-11-20; CC BY 4.0  
**Time:** 20–35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.  
**Author:** JupyterLite-C3S-Atlas contributors (AI assisted) · **Created/updated:** 2026-11-20  
**Inputs:** bundled reference files and/or seeded examples specified below. **Outputs:** tables, figures, and any explicitly named local exports; source data are not modified.  
**Runtime:** Python/Pyodide; NumPy, pandas, Matplotlib, Plotly; versions and tests are recorded in the validation report.  
**Change log:** initial educational chapter. **Contact:** [repository issues](https://github.com/jltobias/JupyterLite-C3S-Atlas/issues).

[Open in JupyterLite Lab](https://jltobias.github.io/JupyterLite-C3S-Atlas/lite/lab/index.html?path=22-pacific-regions.ipynb) · [Official Atlas](https://atlas.climate.copernicus.eu/)


## Read the evidence

The Niño regions sample different parts of the equatorial Pacific. They
        overlap, so their statistics are not independent observations or pieces that can
        be summed into a global average. Their definitions are geographic boxes; the
        producer averages **ocean points** inside them.

        Longitude wraps. Niño 4 spans 160°E to 150°W: its width is 50°, not 310°.
        GeoJSON uses longitude then latitude in degrees. Cut the geometry at ±180°
        so a GIS does not shade the long way around the planet. A rectangle's spherical
        area is proportional to longitude width times the difference of sine latitude;
        that geometric area includes any land and is not the exact producer ocean mask.

## Start the browser lab

Run this setup cell once. First load needs internet for the Python runtime and packages. The remaining lesson uses bundled data. If a 3D view is unavailable, use the 2D/table alternative. Download your edited notebook before clearing browser storage.

In [1]:
# Install only in the browser kernel; desktop dependencies come from requirements.txt.
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "plotly==6.5.0"])

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from atlas_helpers import (
    DATA, SYNTHETIC, COLORS, show, monthly_demo, anomalies, regular_grid,
    area_mean, coastlines, map_figure, globe_figure, warming_levels, load_monthly_csv,
)
print("Ready. Browser edits live in this browser; download work you want to keep.")


Ready. Browser edits live in this browser; download work you want to keep.


## Check geometry independently

In [2]:
from pulse_helpers import (
    load_pulse, complete_months, rank_same_month, region_rings, region_geojson,
    region_map, region_globe, REGIONS, ANOM, REL, CREDIT, BASE,
)
rows = []
for key, r in REGIONS.items():
    width = (r["east"] - r["west"]) % 360
    area = 6371.0**2 * np.deg2rad(width) * (np.sin(np.deg2rad(r["north"])) - np.sin(np.deg2rad(r["south"])))
    rings = region_rings(key)
    assert all(ring[0] == ring[-1] for ring in rings)
    rows.append({"region": r["name"], "longitude width (degrees)": width,
                 "GeoJSON parts": len(rings), "box area (million km²)": area/1e6})
display(pd.DataFrame(rows))
assert rows[-1]["longitude width (degrees)"] == 50

,region,longitude width (degrees),GeoJSON parts,box area (million km²)
0,Niño 1+2,10,1,1.230163
1,Niño 3,60,1,7.409175
2,Niño 3.4,50,1,6.174312
3,Niño 4,50,2,6.174312


## Follow a box across the seam

In [3]:
show(region_map("nino4"))
print(json.dumps(region_geojson()["features"][-1]["geometry"], indent=2))

Figure({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': '#687c8e', 'width': 1},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('YK5FC9B2ZMBLXMe4YmNkwGLaN/fXJ2' ... 'bgYGTADTm2niFiZMBgrkUL0HZkwA=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('nRN7aB+mU8D3OqkvS45TwEBs6dFUmF' ... 'hxu1PAcqjfha23U8CdE3toH6ZTwA=='),
                    'dtype': 'f8'}},
             {'hoverinfo': 'skip',
              'line': {'color': '#687c8e', 'width': 1},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('0XmNXaLKGMDECOHRxiEYwPjhICHKJx' ... 'KFlnXvGsDIemr11aUWwNF5jV2iyhjA'),
                    'dtype': 'f8'},
              'y': {'bdata': ('vhOzXgzvSkCBCdy6m5NKQOuM74tLIU' ... 'THRiCWS0C6oSk7/UZLQL4Ts14M70pA'),
                    'dtype': 'f8'}},
             {'hoverinfo': 'skip',
              'line': {'color': '#687c8e', 'width': 1},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('O+RmuAGgYUBtb7ckh9dhQP9Z8+OvEm' ... 'Hf6mVhQD83NGWnfWFAO+RmuAGgYUA='),
                    'dtype': 'f8'},
              'y': {'bdata': ('J0ut9xvNBMCDUrRyL1AKwF980R4v5A' ... 'PgDWkAwBKDwMqhRQPAJ0ut9xvNBMA='),
                    'dtype': 'f8'}},
             {'hoverinfo': 'skip',
              'line': {'color': '#687c8e', 'width': 1},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('/rRRnQ6NXEAHeNLCZaZcQLt+wW7Y3F' ... 'onuD9cQB9LH7qgbVxA/rRRnQ6NXEA='),
                    'dtype': 'f8'},
              'y': {'bdata': ('TkS/tn4aEkA5Kcx7nJkTQJjArbt5yh' ... 'J4tNEIQAx07QvoJQ9ATkS/tn4aEkA='),
                    'dtype': 'f8'}},
             {'hoverinfo': 'skip',
              'line': {'color': '#687c8e', 'width': 1},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('YRvxZDdnV8Ct3AvMCopXwJUsJ6H05l' ... 'VztlfAxjAnaJN+V8BhG/FkN2dXwA=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('ZktWRbi+UkCPOGQD6aVSQE64V+atql' ... 'Vs6VJA1GUxsfnSUkBmS1ZFuL5SQA=='),
                    'dtype': 'f8'}},
             {'hoverinfo': 'skip',
              'line': {'color': '#687c8e', 'width': 1},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': 'rmLxm8J1V8DnG9E965JXwK1oc5zbCljATMecZ+wbWMCtwmaAC5tXwKPqVzofblfArmLxm8J1V8A=',
                    'dtype': 'f8'},
              'y': {'bdata': 'KUF/oUdhU0CWB+kpcl9TQJepSfCGY1NAxNLAj2p1U0CdY0D2enRTQLL1DOGYaFNAKUF/oUdhU0A=',
                    'dtype': 'f8'}},
             {'hoverinfo': 'skip',
              'line': {'color': '#687c8e', 'width': 1},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('pki+EkgwWMDU1/M1y+NXwBLCo40j9V' ... 'YEbXKoWMCiDFUxlVVYwKZIvhJIMFjA'),
                    'dtype': 'f8'},
              'y': {'bdata': ('gXaHFAOxU0BIG0esxZpTQEwz3eukg1' ... 'l8s823U0DmstE5P7VTQIF2hxQDsVNA'),
                    'dtype': 'f8'}},
             {'hoverinfo': 'skip',
              'line': {'color': '#687c8e', 'width': 1},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('CD2bVZ8JVsAIIos08XBWwFXf+UUJG1' ... 'Uzps5UwII7UKc8hlXACD2bVZ8JVsA='),
                    'dtype': 'f8'},
              'y': {'bdata': ('e7/RjhuZUkCEEmba/qBSQJf9utOdtV' ... 'UIGaRSQBCU2/Y9mlJAe7/RjhuZUkA='),
                    'dtype': 'f8'}},
             {'hoverinfo': 'skip',
              'line': {'color': '#687c8e', 'width': 1},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': '/ilVouzQW8AdAHFXr3Zbw

{
  "type": "MultiPolygon",
  "coordinates": [
    [
      [
        [
          160,
          -5
        ],
        [
          180,
          -5
        ],
        [
          180,
          5
        ],
        [
          160,
          5
        ],
        [
          160,
          -5
        ]
      ]
    ],
    [
      [
        [
          -180,
          -5
        ],
        [
          -150,
          -5
        ],
        [
          -150,
          5
        ],
        [
          -180,
          5
        ],
        [
          -180,
          -5
        ]
      ]
    ]
  ]
}


## Try it and explain

Change the selected region to nino12. Explain why a smaller geographic box can still be scientifically useful.

**Interpretation check:** The boxes sample different ocean regimes. Area alone does not determine relevance, and overlapping boxes cannot be treated as disjoint tiles.

## Transfer to the official tools

Use the official Pulse region selector to compare all four Niño regions. Note the time support and SST units before comparing their temperatures.

Record the selections, source version, units, periods, and uncertainty legend. These independent lessons do not duplicate every production workflow in the official tools.

## Sources and credit

- [C3S Atlas dataset and DOI](https://doi.org/10.24381/cds.h35hb680)
- [CSIC / Predictia Atlas user guide](https://confluence.ecmwf.int/spaces/CKB/pages/394237545/Copernicus+Interactive+Climate+Atlas+User+Guide)
- [Official C3S Atlas User Tools and contributors](https://ecmwf-projects.github.io/c3s-atlas/)
- [C3S Climate Pulse 2.0 announcement (10 November 2026)](https://climate.copernicus.eu/new-version-climate-pulse-provides-new-monitoring-el-nino-conditions)
- [Climate Pulse FAQs and methodology](https://pulse.climate.copernicus.eu/faqs)
- [Climate Pulse credits and citations](https://pulse.climate.copernicus.eu/credits)
- [ERA5 monthly data: C3S/ECMWF and ERA5 contributors](https://doi.org/10.24381/cds.f17050d7)
- [Hersbach et al. (2020): The ERA5 global reanalysis](https://doi.org/10.1002/qj.3803)
- [ECMWF: ENSO region definitions and impacts](https://confluence.ecmwf.int/spaces/CKB/pages/373753799/ENSO+impacts+based+on+ERA5+reanalysis)
- [IETF: GeoJSON geometry and antimeridian guidance](https://www.rfc-editor.org/rfc/rfc7946)

Original teaching text and code; original AI-generated splash illustration. Real reference files retain their source licences. Synthetic examples are not Copernicus measurements. See [credits](credits.md) and [data guide](data-guide.md).

**Continue:** [Warm water, unusual water](23-pulse-baselines.ipynb)